In [ ]:
import pickle

import sys
sys.path.append('..')

import jax
import jax.numpy as jnp
import numpy as np
from matplotlib import pyplot as plt

from lucid.geometry import generate_detector
from lucid.generate import read_photon_data_from_photonsim
from lucid.optimization.grid_search import load_optimization_config
from lucid.detector_params import ParticleParams, load_detector_params
from lucid.losses import first_arrival_nll, segment_logsumexp

from jax.scipy.special import gammaln

print("Imports successful")

In [ ]:
from lucid.simulation import setup_event_simulator

# =====================================================================
# Full script: 2D parameter scans + optimization trajectory overlay
# =====================================================================
import time
import pickle
from pathlib import Path
import numpy as np
import jax
import jax.numpy as jnp
from jax import value_and_grad
from tqdm import tqdm
import matplotlib.pyplot as plt

# =====================================================================
# Configuration
# =====================================================================
default_json_filename = '../config/SK_geom_config.json'
PHYSICS_CONFIG = '../config/SK_physics_config.json'
TEMPERATURE = 0.10
N_SCAN_POINTS = 11
K = 7
Nphot = 150_000

C_MEDIUM = 0.299792 / 1.33  # speed of light in medium

# =====================================================================
# Detector setup
# =====================================================================
detector = generate_detector(default_json_filename)
detector_points = jnp.array(detector.all_points)
detector_radius = detector.S_radius
NUM_DETECTORS = len(detector_points)

prediction_simulator = setup_event_simulator(
    default_json_filename, Nphot, TEMPERATURE, 
    max_sensors_per_cell=4, K=K, is_data=False, hit_mode='per_photon',
    physics_config=PHYSICS_CONFIG, default_detector_params=True
)

# Setup data simulator for generating target events (is_data=True, temperature=0.0)
data_simulator = setup_event_simulator(default_json_filename, Nphot, temperature=0.0, K=20,
                                      is_data=True, is_calibration=False,
                                      physics_config=PHYSICS_CONFIG, default_detector_params=True)

In [ ]:
import jax
import jax.numpy as jnp
import time

from jax import jit
from pathlib import Path

from matplotlib import pyplot as plt
plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12

import numpy as np
from functools import partial
import pickle
from tqdm import tqdm
from jax import grad, jit, vmap, value_and_grad
import uproot
from scipy.interpolate import interp1d

from lucid.optimization.utils.functions import spherical_to_cartesian, cartesian_to_spherical

# =====================================================================
# Likelihood-based loss with vertex term and dynamic tau_vtx
# =====================================================================
from lucid.losses import (
    first_arrival_nll,
    poisson_nll,
    origin_time_loss_configurable,
    TAU_VTX_PARAM_A,
    TAU_VTX_PARAM_B,
    TAU_VTX_PARAM_C,
)

TAU_TIME = 0.15  # Fixed tau for first_arrival_nll
NRAYS_FLOAT = float(Nphot)  # From Cell 2


@jit
def combined_product_loss(params, observed_times, observed_counts, key):
    """
    Likelihood-based loss: 3-term geometric mean of charge, time, and vertex losses.

    Args:
        params: [x, y, z, t0, theta, phi, energy]

    Returns:
        combined_loss, (charge_loss, time_loss, vertex_loss)
    """
    position = params[:3]
    t0 = params[3]
    theta = params[4]
    phi = params[5]
    energy = params[6]

    # Simulate with t0=0
    track = ParticleParams(energy=energy, position=position, theta=theta, phi=phi, t0=jnp.array(0.0))
    log_w, flat_times, flat_indices, total_charge = prediction_simulator(track, key)

    # Charge loss (Poisson NLL)
    charge_loss = poisson_nll(observed_counts, total_charge)

    # Time loss (First-arrival NLL with shifted observations)
    t_obs_shifted = observed_times - t0
    time_nll = first_arrival_nll(
        log_w, flat_times, flat_indices,
        t_obs_shifted, TAU_TIME, NUM_DETECTORS)

    hit_mask = observed_counts > 0
    n_hit = jnp.sum(hit_mask) + 1e-8
    time_loss = jnp.sum(jnp.where(hit_mask, time_nll, 0.0)) / n_hit

    # Vertex loss with dynamic tau_vtx
    tau_vtx = jax.lax.stop_gradient(
        TAU_VTX_PARAM_A * NRAYS_FLOAT + TAU_VTX_PARAM_B * energy + TAU_VTX_PARAM_C
    )
    tau_vtx = jnp.clip(tau_vtx, 0.05, 0.95)

    vertex_loss = origin_time_loss_configurable(
        jax.lax.stop_gradient(position), detector_points,
        observed_times, observed_counts, t0, tau=tau_vtx
    )

    # 3-term combined loss
    c, t, v, s = charge_loss, time_loss, vertex_loss, 0.
    combined = (jnp.sqrt((c + s) * (t + s) * (v + s)) +
                jnp.sqrt((c + s) * jax.lax.stop_gradient((t + s) * (v + s))) +
                jnp.sqrt((v + s) * jax.lax.stop_gradient((t + s) * (c + s))))

    return combined, (charge_loss, time_loss, vertex_loss)


@jit
def combined_product_loss_landscape(params, observed_times, observed_counts, key):
    """Wrapper returning only the combined loss (for optimization landscapes)."""
    combined_loss, _ = combined_product_loss(params, observed_times, observed_counts, key)
    return combined_loss

import json
import subprocess
from lucid.optimization.grid_search import get_detector_bounds

def load_config(config_path):
    """Load and validate configuration file"""
    config_path = Path(config_path)
    if not config_path.exists():
        raise FileNotFoundError(f"Configuration file not found: {config_path}")

    config = load_optimization_config(str(config_path))

    # Add default values for new parameters if not present
    if 'optimization_params' not in config:
        config['optimization_params'] = {}

    optimization_params = config['optimization_params']
    optimization_params.setdefault('damping_factor', 1.000)

    # Add Adam optimizer parameters if not present
    if 'adam_optimizer' not in config:
        config['adam_optimizer'] = {}

    adam_params = config['adam_optimizer']
    adam_params.setdefault('learning_rate', 0.1)
    adam_params.setdefault('b1', 0.95)
    adam_params.setdefault('b2', 0.99)
    adam_params.setdefault('eps', 1e-8)

    return config

combined_grad_fn = jit(value_and_grad(combined_product_loss, has_aux=True))

CONFIG_INDEX = 4
config_dir = Path('../s3df_jobs/nrays_config')
config_path = config_dir / f'opt_config_{CONFIG_INDEX}.json'
script_path = config_dir / 'create_configs.py'

if not config_path.exists():
    print("Config file not found. Creating it...")
    subprocess.run(
        [sys.executable, script_path.name],
        cwd=config_dir,
        check=True
    )
    print("Config file successfully created.")
else:
    print("Config file already exists.")

adam_config = load_config(str(config_path))
detector_bounds = get_detector_bounds(detector)


print("Likelihood loss function defined (with vertex term and dynamic tau_vtx)")

# Detector parameters
detector_params = load_detector_params(PHYSICS_CONFIG)

In [ ]:
data_file = '../data/water/muon/muon_gun_1050_MeV_100_events_fixed_energy.root'

# Select entry from ROOT file
entry_idx = 1

# Load photon data from ROOT file
photon_data = read_photon_data_from_photonsim(data_file, entry_idx)

# Process photon data
photon_origins = photon_data['photon_origins']
photon_directions = photon_data['photon_directions']
photon_times = photon_data['photon_times']
N = len(photon_origins)

# the number 1_000_000 is hard coded also in _simulation_core
padding_size = max(0, 1_000_000-N)

# Pad the origins array (2D array with shape [N,3])
photon_data['photon_origins'] = jnp.pad(photon_origins, ((0, padding_size), (0, 0)), 
                                    mode='constant', constant_values=0)

# Pad the directions array with a default unit vector [0,0,1]
default_direction = jnp.array([0.0, 0.0, 1.0])
padding_directions = jnp.tile(default_direction, (padding_size, 1))
if padding_size > 0:
    photon_data['photon_directions'] = jnp.concatenate([photon_directions, padding_directions], axis=0)
else:
    photon_data['photon_directions'] = photon_directions

# Pad the times array (1D array with shape [N])
photon_data['photon_times'] = jnp.pad(photon_times, (0, padding_size),
                                      mode='constant', constant_values=0)

photon_data['N'] = N

# Generate random track parameters
key = jax.random.PRNGKey(45)

# Random position within detector bounds (60% of full volume)
fraction = 0.6
r_vert = jax.random.uniform(key, shape=(), minval=0, maxval=detector.r * fraction)
key, _ = jax.random.split(key)
theta_pos = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
key, _ = jax.random.split(key)
z_vert = jax.random.uniform(key, shape=(), minval=-detector.H/2 * fraction, 
                           maxval=detector.H/2 * fraction)
true_position = jnp.array([r_vert * jnp.cos(theta_pos), r_vert * jnp.sin(theta_pos), z_vert])

# Random direction
key, _ = jax.random.split(key)
phi = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
key, _ = jax.random.split(key)
cos_theta = jax.random.uniform(key, shape=(), minval=-1, maxval=1)
sin_theta = jnp.sqrt(1 - cos_theta**2)
true_direction = jnp.array([sin_theta * jnp.cos(phi), sin_theta * jnp.sin(phi), cos_theta])

# Use energy from ROOT file
true_energy = photon_data['energy']  # Fixed energy

# Create particle parameters
true_track = ParticleParams.from_cartesian(energy=true_energy, position=true_position, direction=true_direction, t0=0.0)

# Compute rotation to transform from original direction (0,0,1) to true_direction
original_direction = jnp.array([0.0, 0.0, 1.0])
true_direction_norm = true_direction / (jnp.linalg.norm(true_direction) + 1e-8)

# Rotation axis = cross product of original and target directions
rotation_axis = jnp.cross(original_direction, true_direction_norm)
axis_norm = jnp.linalg.norm(rotation_axis)

# Handle case where directions are parallel (axis_norm ~ 0)
rotation_axis = jnp.where(
    axis_norm < 1e-6,
    jnp.array([1.0, 0.0, 0.0]),  # Arbitrary axis when parallel
    rotation_axis / (axis_norm + 1e-8)
)

# Rotation angle = arccos of dot product
rotation_angle = jnp.arccos(jnp.clip(
    jnp.dot(original_direction, true_direction_norm), -1.0, 1.0
))

# Set rotation parameters in photon_data
photon_data['rotation_axis'] = rotation_axis
photon_data['rotation_angle'] = rotation_angle
photon_data['apply_rotation'] = jnp.array(True)

# Set translation parameters to move from origin to true_position
photon_data['apply_translation'] = jnp.array(True)
photon_data['translation_vector'] = true_position

# Generate data-like event
key, _ = jax.random.split(key)
true_data = jax.lax.stop_gradient(data_simulator(true_track, key, photon_data))

# Convert true direction (cartesian -> spherical)
x, y, z = true_direction
true_theta = np.mod(np.arccos(z),  np.pi)
true_phi =   np.mod(np.arctan2(y, x), 2 * np.pi)

key, _ = jax.random.split(key)
true_t0 = jax.random.uniform(key, shape=(), minval=-15.0, maxval=15.0)

hit_counts, hit_times = true_data
hit_times += true_t0

true_data = [hit_counts, hit_times]

# Construct parameter array [x, y, z, t0, theta, phi, energy]
true_param_values = [
    float(true_position[0]),
    float(true_position[1]),
    float(true_position[2]),
    float(true_t0),
    float(true_theta),
    float(true_phi),
    float(true_energy)
]

# =====================================================================
# Define parameter ranges and pairs
# =====================================================================
param_configs = {
    0: ('X', 2.0),
    1: ('Y', 2.0),
    2: ('Z', 2.0),
    3: ('t0', 4.0),
    4: ('theta', 0.5),
    5: ('phi', 0.5),
    6: ('E', 100.0),
}

scan_pairs = [
    (0, 2),  # (X, Z)
    (1, 5),  # (Y, phi)
    (5, 6),  # (phi, E)
    (0, 3),  # (X, t0)
    (3, 5),  # (t0, phi)
    (3, 6)   # (t0, E)
]

print(f"Total 2D scans: {len(scan_pairs)}  ({len(scan_pairs) * N_SCAN_POINTS**2} total loss evals)")

# =====================================================================
# Define loss function used in scan
# =====================================================================
def perform_2d_parameter_scan(true_track, true_data, key, param1_name, param1_idx, param1_range,
                             param2_name, param2_idx, param2_range, true_param_values, 
                             use_relative=True, verbose=False):
    """Perform 2D scan of loss vs. two parameters."""
    t_start_total = time.time()
    hit_counts, hit_times = true_data
    observed_times = hit_times
    observed_charge = hit_counts

    true_value1 = true_param_values[param1_idx]
    true_value2 = true_param_values[param2_idx]

    if use_relative:
        param1_values = jnp.linspace(true_value1 - param1_range, true_value1 + param1_range, N_SCAN_POINTS)
        param2_values = jnp.linspace(true_value2 - param2_range, true_value2 + param2_range, N_SCAN_POINTS)
    else:
        param1_values = jnp.linspace(param1_range[0], param1_range[1], N_SCAN_POINTS)
        param2_values = jnp.linspace(param2_range[0], param2_range[1], N_SCAN_POINTS)

    def loss_and_grad_fn(params):
        def loss_fn(p):
            loss_key = jax.random.PRNGKey(42)
            return combined_product_loss_landscape(p, observed_times, observed_charge, loss_key)
        return value_and_grad(loss_fn)(params)

    warmup_params = jnp.array(true_param_values)
    _ = loss_and_grad_fn(warmup_params)
    jax.block_until_ready(_)

    losses = jnp.zeros((N_SCAN_POINTS, N_SCAN_POINTS))
    gradients = jnp.zeros((N_SCAN_POINTS, N_SCAN_POINTS, 2))
    t_start_scan = time.time()

    for i, p1 in enumerate(tqdm(param1_values, desc=f"Scanning {param1_name}")):
        for j, p2 in enumerate(param2_values):
            params = jnp.array(true_param_values)
            params = params.at[param1_idx].set(p1)
            params = params.at[param2_idx].set(p2)
            loss, grad_val = loss_and_grad_fn(params)
            jax.block_until_ready((loss, grad_val))
            losses = losses.at[i, j].set(loss)
            gradients = gradients.at[i, j].set(jnp.array([grad_val[param1_idx], grad_val[param2_idx]]))

    timing = time.time() - t_start_scan
    if verbose:
        print(f"  Scan time: {timing:.2f}s")

    return {
        'param1_name': param1_name,
        'param2_name': param2_name,
        'param1_values': np.array(param1_values),
        'param2_values': np.array(param2_values),
        'losses': np.array(losses),
        'gradients': np.array(gradients),
        'true_value1': float(true_value1),
        'true_value2': float(true_value2),
    }

# =====================================================================
# Perform all 2D scans
# =====================================================================
key = jax.random.PRNGKey(42)
true_track = ParticleParams.from_cartesian(energy=true_energy, position=true_position, direction=true_direction, t0=0.0)

scan_results_2d = []
for i, (p1_idx, p2_idx) in enumerate(scan_pairs):
    p1_name, p1_range = param_configs[p1_idx]
    p2_name, p2_range = param_configs[p2_idx]
    print(f"\n[{i+1}/{len(scan_pairs)}] ({p1_name}, {p2_name})")
    key, _ = jax.random.split(key)
    result = perform_2d_parameter_scan(
        true_track, true_data, key,
        p1_name, p1_idx, p1_range,
        p2_name, p2_idx, p2_range,
        true_param_values, use_relative=True, verbose=True
    )
    scan_results_2d.append(result)
# =====================================================================
# Save results
# =====================================================================
output_dir = Path('../output')
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / f'event_{entry_idx}_2D_scans_likelihood.pkl'
with open(output_file, 'wb') as f:
    pickle.dump(scan_results_2d, f)
print(f"Saved: {output_file}")

In [ ]:
def run_complete_optimization_adam_from_guess(
    initial_params,  # [x, y, z, t0, theta, phi, energy]
    observed_times,
    observed_counts,
    true_data,
    true_energy,
    true_position,
    true_direction,
    TRUE_T0,
    config,
    detector_bounds,
    prediction_simulator,
    combined_grad_fn,
    verbosity=2
):
    """
    Run Adam optimization starting from a provided initial guess.
    Uses likelihood-based loss (Poisson NLL + first-arrival NLL + vertex loss).
    
    Matches the optimizer settings from tracking_opt_development_likelihood.ipynb.
    """

    import jax
    import jax.numpy as jnp
    import numpy as np
    import time
    from jax import value_and_grad
    import optax
    

    # Unpack true spherical coords for error metrics
    true_theta, true_phi = cartesian_to_spherical(true_direction)

    # Extract optimizer configuration
    ADAM_LEARNING_RATE = config['adam_optimizer']['learning_rate']
    ADAM_B1 = config['adam_optimizer']['b1']
    ADAM_B2 = config['adam_optimizer']['b2']
    ADAM_EPS = config['adam_optimizer']['eps']
    MAX_ITERATIONS = 600#config['gradient_descent']['max_iterations']
    tolerance = 1e-6

    # Learning-rate scaling (matching reference notebook)
    POS_LR_SCALE = config['learning_rates']['position_learning_rate'] * 2.
    DIR_LR_SCALE = config['learning_rates']['direction_learning_rate'] * 10.
    T0_LR_SCALE = config['learning_rates']['t0_learning_rate']
    ENE_LR_SCALE = config['learning_rates']['energy_learning_rate']

    # Detector constraints
    DETECTOR_R = detector_bounds.get('r', None)
    DETECTOR_H = detector_bounds.get('H', None)

    if verbosity >= 2:
        print("\nStarting Adam optimization from provided initial parameters:")
        print(f"  Initial params: {initial_params}")
        print(f"  True position: {true_position}")
        print(f"  True direction: {true_direction}")
        print(f"  True energy: {true_energy:.1f} MeV, True t0: {TRUE_T0:.3f}")
        print(f"  Learning rate: {ADAM_LEARNING_RATE}")
        print(f"  Max iterations: {MAX_ITERATIONS}")

    # Initialize Adam optimizer
    optimizer = optax.adam(learning_rate=ADAM_LEARNING_RATE, b1=ADAM_B1, b2=ADAM_B2, eps=ADAM_EPS)
    opt_state = optimizer.init(initial_params)
    current_params = jnp.array(initial_params)

    history = {
        'parameters': [current_params.copy()],
        'combined_losses': [],
        'charge_losses': [],
        'time_losses': [],
        'vertex_losses': [],
        'position_errors': [],
        'direction_errors': [],
        't0_errors': [],
        'energy_errors': [],
    }

    opt_key = jax.random.PRNGKey(12345)
    grad_norm = float('inf')

    adam_start_time = time.time()

    for iteration in range(MAX_ITERATIONS):
        opt_key, _ = jax.random.split(opt_key)

        (combined_loss, (charge_loss_val, time_loss_val, vertex_loss_val)), grad = combined_grad_fn(
            current_params, observed_times, observed_counts, opt_key
        )

        if jnp.any(jnp.isnan(grad)):
            grad = jnp.nan_to_num(grad, nan=0.0)

        grad_norm = jnp.linalg.norm(grad)
        if grad_norm < tolerance:
            break

        # Phase 1: Direction only for first 25 iterations (matching reference)
        if iteration < 25:
            update_scales = jnp.array([0, 0, 0, 0, DIR_LR_SCALE, DIR_LR_SCALE, 0.])
        else:
            update_scales = jnp.array([
                POS_LR_SCALE, POS_LR_SCALE, POS_LR_SCALE,
                T0_LR_SCALE, DIR_LR_SCALE, DIR_LR_SCALE, ENE_LR_SCALE
            ])

        # Adam update with parameter-specific scaling (NO damping, matching reference)
        updates, opt_state = optimizer.update(grad, opt_state, current_params)
        scaled_updates = updates * update_scales
        # if iteration < 500:
        #     scaled_updates /= 4    
        
        current_params = optax.apply_updates(current_params, scaled_updates)

        # Clip within detector bounds
        if DETECTOR_R is not None and DETECTOR_H is not None:
            current_params = jnp.array([
                jnp.clip(current_params[0], -DETECTOR_R * 0.95, DETECTOR_R * 0.95),
                jnp.clip(current_params[1], -DETECTOR_R * 0.95, DETECTOR_R * 0.95),
                jnp.clip(current_params[2], -DETECTOR_H/2 * 0.95, DETECTOR_H/2 * 0.95),
                jnp.clip(current_params[3], -20.0, 20.0),
                current_params[4],
                current_params[5],
                jnp.clip(current_params[6], 300.0, 2000.0)
            ])

        # Calculate derived quantities
        current_position = current_params[:3]
        current_t0 = current_params[3]
        current_theta = current_params[4]
        current_phi = current_params[5]
        current_energy = current_params[6]
        current_direction = spherical_to_cartesian(current_theta, current_phi)

        # Errors
        position_error = float(jnp.linalg.norm(current_position - true_position))
        energy_error = float(abs(current_energy - true_energy))
        t0_error = float(abs(current_t0 - TRUE_T0))
        cos_angle = np.clip(np.dot(np.array(current_direction), np.array(true_direction)), -1.0, 1.0)
        direction_error = float(np.degrees(np.arccos(cos_angle)))

        # Store
        history['parameters'].append(current_params.copy())
        history['combined_losses'].append(float(combined_loss))
        history['charge_losses'].append(float(charge_loss_val))
        history['time_losses'].append(float(time_loss_val))
        history['vertex_losses'].append(float(vertex_loss_val))
        history['position_errors'].append(position_error)
        history['direction_errors'].append(direction_error)
        history['t0_errors'].append(t0_error)
        history['energy_errors'].append(energy_error)

        if verbosity >= 2 and ((iteration + 1) % 100 == 0 or iteration == 0):
            print(f"  Iter {iteration}: loss={combined_loss:.6f}, "
                  f"(c={charge_loss_val:.4f}, t={time_loss_val:.4f}, v={vertex_loss_val:.4f}) "
                  f"pos_err={position_error:.3f}m, dir_err={direction_error:.2f}°, "
                  f"t0_err={t0_error:.3f}, E_err={energy_error:.1f}")

    adam_end_time = time.time()

    # Final state
    final_position = current_params[:3]
    final_t0 = current_params[3]
    final_theta = current_params[4]
    final_phi = current_params[5]
    final_energy = current_params[6]
    final_direction = spherical_to_cartesian(final_theta, final_phi)

    cos_angle_final = np.clip(np.dot(np.array(final_direction), np.array(true_direction)), -1.0, 1.0)
    final_direction_error = float(np.degrees(np.arccos(cos_angle_final)))
    final_position_error = float(jnp.linalg.norm(final_position - true_position))
    final_energy_error = float(abs(final_energy - true_energy))
    final_t0_error = float(abs(final_t0 - TRUE_T0))

    if verbosity >= 2:
        print(f"\n  Optimization completed in {adam_end_time - adam_start_time:.2f}s ({iteration+1} iterations)")
        print(f"  Final position error: {final_position_error:.3f} m")
        print(f"  Final direction error: {final_direction_error:.2f} deg")
        print(f"  Final t0 error: {final_t0_error:.3f}")
        print(f"  Final energy error: {final_energy_error:.1f} MeV")

    return {
        'initial_params': np.array(initial_params),
        'final_params': np.array(current_params),
        'final_position': np.array(final_position),
        'final_direction': np.array(final_direction),
        'final_theta': float(final_theta),
        'final_phi': float(final_phi),
        'final_t0': float(final_t0),
        'final_energy': float(final_energy),
        'final_position_error': final_position_error,
        'final_direction_error': final_direction_error,
        'final_t0_error': final_t0_error,
        'final_energy_error': final_energy_error,
        'adam_optimization_time': adam_end_time - adam_start_time,
        'history': history,
        'converged': grad_norm < tolerance,
    }

In [ ]:
import numpy as np

def generate_initial_guess_from_true(
    true_params,
    position_sigma=0.5,   # meters
    t0_sigma=5.0,         # ns
    theta_sigma=np.radians(5.0),  # radians
    phi_sigma=np.radians(5.0),    # radians
    energy_sigma=100.0,   # MeV
    detector_bounds=None,
    energy_bounds=(300.0, 2000.0),
    seed=None
):
    """
    Create a random Gaussian-shifted initial guess around the true parameters.

    Args:
        true_params (array-like): [X, Y, Z, t0, theta, phi, energy]
        position_sigma (float): std. dev. of position perturbation (m)
        t0_sigma (float): std. dev. of t0 perturbation (ns)
        theta_sigma (float): std. dev. of theta perturbation (rad)
        phi_sigma (float): std. dev. of phi perturbation (rad)
        energy_sigma (float): std. dev. of energy perturbation (MeV)
        detector_bounds (dict, optional): {'r': ..., 'H': ...} cylindrical bounds
        energy_bounds (tuple): (min, max) for energy clipping
        seed (int, optional): random seed for reproducibility

    Returns:
        np.ndarray: perturbed initial guess [X, Y, Z, t0, theta, phi, energy]
    """
    rng = np.random.default_rng(seed)

    X, Y, Z, t0, theta, phi, E = np.array(true_params, dtype=float)

    # Gaussian perturbations
    X_guess = X + rng.normal(0, position_sigma)
    Y_guess = Y + rng.normal(0, position_sigma)
    Z_guess = Z + rng.normal(0, position_sigma)
    t0_guess = t0 + rng.normal(0, t0_sigma)
    theta_guess = theta + rng.normal(0, theta_sigma)
    phi_guess = phi + rng.normal(0, phi_sigma)
    E_guess = E + rng.normal(0, energy_sigma)

    # Clip phi to [0, 2π)
    phi_guess = np.mod(phi_guess, 2 * np.pi)

    # Clip theta to [0, π]
    theta_guess = np.clip(theta_guess, 0, np.pi)

    # Enforce detector and energy limits
    if detector_bounds is not None:
        R = detector_bounds.get('r', None)
        H = detector_bounds.get('H', None)
        if R is not None:
            X_guess = np.clip(X_guess, -R * 0.95, R * 0.95)
            Y_guess = np.clip(Y_guess, -R * 0.95, R * 0.95)
        if H is not None:
            Z_guess = np.clip(Z_guess, -H/2 * 0.95, H/2 * 0.95)

    E_guess = np.clip(E_guess, *energy_bounds)
    t0_guess = np.clip(t0_guess, -20.0, 20.0)

    return np.array([X_guess, Y_guess, Z_guess, t0_guess, theta_guess, phi_guess, E_guess])


In [ ]:
true_params = np.array([
    true_position[0],
    true_position[1],
    true_position[2],
    true_t0,
    true_theta,
    true_phi,
    true_energy
])

all_event_results = []
for i in range(3):
    initial_guess = generate_initial_guess_from_true(
        true_params,
        position_sigma=1.0,   # 100 cm deviation
        t0_sigma=3.0,         # 3 ns deviation
        theta_sigma=np.radians(10.0),  # 10°
        phi_sigma=np.radians(10.0),    # 10°
        energy_sigma=100.0,    # 100 MeV
        detector_bounds=detector_bounds,
        seed=42+i
    )
    
    print("True params:    ", np.round(true_params, 3))
    print("Initial guess:  ", np.round(initial_guess, 3))
    
    results_from_guess = run_complete_optimization_adam_from_guess(
        initial_params=initial_guess,
        observed_times=true_data[1],
        observed_counts=true_data[0],
        true_data=true_data,
        true_energy=true_energy,
        true_position=true_position,
        true_direction=true_direction,
        TRUE_T0=true_t0,
        config=adam_config,
        detector_bounds=detector_bounds,
        prediction_simulator=prediction_simulator,
        combined_grad_fn=combined_grad_fn,
    )
    all_event_results.append(results_from_guess)

In [ ]:
# =====================================================================
# Reconstruct parameter evolution (from optimization history)
# =====================================================================
def extract_histories(all_event_results):
    """
    Collect all optimization histories from multiple initial guesses.
    Compatible with results returned by run_complete_optimization_adam_from_guess().
    """
    h = {'position': [], 'direction': [], 'energy': [], 't0': []}

    for ev in all_event_results:
        hist = ev['history']
        params = np.array(hist['parameters'])  # shape (n_iter, 7)
        # Columns: [x, y, z, t0, theta, phi, energy]
        h['position'].append(params[:, :3])
        h['t0'].append(params[:, 3])
        h['direction'].append(params[:, 4:6])  # theta, phi
        h['energy'].append(params[:, 6])

    # Convert lists to numpy arrays
    for k in h:
        h[k] = np.array(h[k])  # (n_events, n_iterations, ...)
    return h


histories = extract_histories(all_event_results)


def reconstruct_reco_parameters(all_event_results, histories, true_position, true_direction, true_energy, true_t0):
    """
    Build reco parameter arrays aligned with the true parameters.
    Works for multiple events (one per initial guess) or just one.
    """
    n_events, n_iterations = histories['t0'].shape[:2]

    # Convert true direction to spherical
    x, y, z = true_direction
    true_theta = np.arccos(z)
    true_phi = np.arctan2(y, x)

    # Tile true values for vectorized reconstruction
    true_pos = np.tile(true_position, (n_iterations, 1))
    true_pos = np.expand_dims(true_pos, 0).repeat(n_events, axis=0)
    true_theta_arr = np.full((n_events, n_iterations), true_theta)
    true_phi_arr = np.full((n_events, n_iterations), true_phi)
    true_energy_arr = np.full((n_events, n_iterations), true_energy)
    true_t0_arr = np.full((n_events, n_iterations), true_t0)

    # Construct differences (reco - true)
    position_diff = histories['position'] - true_pos
    direction_diff = histories['direction'] - np.stack([true_theta_arr, true_phi_arr], axis=-1)
    energy_diff = histories['energy'] - true_energy_arr
    t0_diff = histories['t0'] - true_t0_arr

    reco = {
        'position': histories['position'],
        'theta': histories['direction'][:, :, 0],
        'phi': histories['direction'][:, :, 1],
        'energy': histories['energy'],
        't0': histories['t0'],
        'position_diff': position_diff,
        'direction_diff': direction_diff,
        'energy_diff': energy_diff,
        't0_diff': t0_diff
    }
    return reco


# Construct reco trajectories relative to true parameters
reco_data = reconstruct_reco_parameters(
    all_event_results,
    histories,
    true_position=true_position,
    true_direction=true_direction,
    true_energy=true_energy,
    true_t0=true_t0
)

print("Reconstructed reco parameter trajectories.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

MIN_LOSS = float(min([min((scan_data['losses']).flatten()) for scan_data in scan_results_2d]))

def plot_all_gradient_scans_grid(scan_results_2d, reco_data,
                                 save_fig=True, cmap_name='Oranges',
                                 vmin=1.7, vmax=2.2):
    """
    Plot all 2D gradient scans (e.g. 6 scans) in a 2x3 grid,
    overlaying all optimization trajectories with shared colormap scaling
    and a single well-positioned colorbar.

    Args:
        scan_results_2d: list of scan dicts (from perform_2d_parameter_scan)
        reco_data: dict of reconstructed parameter trajectories
        save_fig: whether to save figure
        cmap_name: matplotlib colormap name for paths (default 'Oranges')
        vmin, vmax: global min/max for loss color scale
    """
    n_scans = len(scan_results_2d)
    n_rows, n_cols = 2, 3
    cmap = plt.get_cmap(cmap_name)
    n_events = len(reco_data['t0'])
    colors = [cmap(0.3 + 0.6 * (i / max(n_events - 1, 1))) for i in range(n_events)]

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 6))
    axes = axes.flatten()

    # --- Label formatting helper
    def format_label(name):
        if name == 'theta': return '$\\theta$ (rad)'
        if name == 'phi': return '$\\phi$ (rad)'
        if name == 't0': return '$t_0$ (ns)'
        if name == 'E': return 'E (MeV)'
        if name in ['X', 'Y', 'Z']: return f'{name} (m)'
        return name

    # --- Param mapping g
    param_map = {
        'X': ('position', 0), 'Y': ('position', 1), 'Z': ('position', 2),
        't0': ('t0', None), 'theta': ('theta', None),
        'phi': ('phi', None), 'E': ('energy', None)
    }

    def get_param_values(name, event_id):
        field, idx = param_map[name]
        arr = reco_data[field][event_id]
        return arr if idx is None else arr[:, idx]

    # Swap top-right (index 2) with bottom-middle (index 4)
    axes[2], axes[4] = axes[4], axes[2]
    
    # --- Loop over all scans
    img_handles = []
    for i, (scan_data, ax) in enumerate(zip(scan_results_2d, axes)):
        # --- Extract data
        param1_name = scan_data['param1_name']
        param2_name = scan_data['param2_name']
        param1_values = np.array(scan_data['param1_values'])
        param2_values = np.array(scan_data['param2_values'])
        losses = np.array(scan_data['losses'])
        losses -= min(losses.flatten())
        gradients = np.array(scan_data['gradients'])
        true_value1 = scan_data['true_value1']
        true_value2 = scan_data['true_value2']

        # --- Gradient scaling
        if param1_name in ['X', 'Y', 'Z']:
            gradients[:, :, 0] *= 0.05
        if param2_name in ['X', 'Y', 'Z']:
            gradients[:, :, 1] *= 0.05
        if param1_name in ['theta', 'phi']:
            gradients[:, :, 0] *= 0.005
        if param2_name in ['theta', 'phi']:
            gradients[:, :, 1] *= 0.005
        if param1_name == 't0':
            gradients[:, :, 0] *= 0.1
        if param2_name == 'E':
            gradients[:, :, 1] *= 1000

        # --- Grid
        p1_min, p1_max = float(param1_values[0]), float(param1_values[-1])
        p2_min, p2_max = float(param2_values[0]), float(param2_values[-1])
        n_p1, n_p2 = len(param1_values), len(param2_values)
        p1_lin = np.linspace(p1_min, p1_max, n_p1)
        p2_lin = np.linspace(p2_min, p2_max, n_p2)

        # --- Normalize gradients
        U = -gradients[:, :, 0].T
        V = -gradients[:, :, 1].T
        norm = np.sqrt(U**2 + V**2)
        norm = np.where(norm == 0, 1, norm)
        U /= norm
        V /= norm

        # --- Loss map with fixed vmin/vmax
        im = ax.imshow(
            losses.T, extent=[p1_min, p1_max, p2_min, p2_max],
            origin='lower', aspect='auto', cmap='viridis',
            vmin=vmin, vmax=vmax
        )
        img_handles.append(im)

        # --- Streamlines
        ax.streamplot(p1_lin, p2_lin, U, V, color='white',
                      density=1.0, linewidth=0.7, arrowsize=0.5)

        # --- Overlay trajectories
        for event_id, color in enumerate(colors):
            p1_path = get_param_values(param1_name, event_id)
            p2_path = get_param_values(param2_name, event_id)
            ax.plot(p1_path, p2_path, color=color, linewidth=2, alpha=1.)
            ax.scatter(p1_path, p2_path, color=color, s=10, alpha=0.8)
            ax.plot(p1_path[-1], p2_path[-1], color='cyan',
                    marker='x', markersize=14, zorder=100+event_id)

        # --- True value
        ax.plot(true_value1, true_value2, color='deeppink',
                marker='*', markersize=20, zorder=99)

        ax.set_xlabel(format_label(param1_name))
        ax.set_ylabel(format_label(param2_name))
        ax.set_xlim(p1_min, p1_max)
        ax.set_ylim(p2_min, p2_max)
        #ax.set_title(f"{param1_name} vs {param2_name}")

    # --- Unified colorbar on the right
    cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])  # [left, bottom, width, height]
    cbar = fig.colorbar(img_handles[0], cax=cbar_ax)
    cbar.set_label('$\\Delta$ Loss', fontsize=12)

    # --- Shared legend on top
    legend_elements = [
        plt.Line2D([], [], color=colors[-2], lw=2, label='Optimization paths'),
        plt.Line2D([], [], color='cyan', marker='x', lw=0, markersize=10, label='Reconstructed'),
        plt.Line2D([], [], color='deeppink', marker='*', lw=0, markersize=12, label='True')
    ]
    fig.legend(handles=legend_elements, loc='upper center',
               ncol=3, frameon=False, fontsize=13)

    #plt.tight_layout(rect=[0, 0, 0.9, 0.95])  # leave space for colorbar + legend
    plt.subplots_adjust(
        left=0.05,   # margin from left edge of figure
        right=0.90,  # margin before colorbar (since you use one)
        top=0.92,    # leave room for legend
        bottom=0.08, # bottom margin
        wspace=0.34, # horizontal space between subplots
        hspace=0.32  # vertical space between subplots
    )
    
    if save_fig:
        figures_dir = Path('figures')
        figures_dir.mkdir(parents=True, exist_ok=True)
        figname = figures_dir / '2D_Grad_All_Pairs_Grid.pdf'
        plt.savefig(figname, dpi=150, bbox_inches='tight')
        print(f"Grid figure saved as: {figname}")

    plt.show()

MAX_LOSS = float(max([max((scan_data['losses']).flatten()) for scan_data in scan_results_2d]))
MIN_loss = float(min([min((scan_data['losses']).flatten()) for scan_data in scan_results_2d]))
plot_all_gradient_scans_grid(scan_results_2d, reco_data, save_fig=True, cmap_name='Oranges', vmin=0., vmax=(MAX_LOSS-MIN_LOSS)/2)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import LogNorm


def plot_all_gradient_scans_grid(scan_results_2d, reco_data,
                                 save_fig=True,
                                 cmap_name='viridis',
                                 vmin=1e-6, vmax=1.0):
    """
    Plot all 2D gradient scans in a 2x3 grid with LOG color scale.
    """

    n_scans = len(scan_results_2d)
    n_rows, n_cols = 2, 3
    cmap = plt.get_cmap(cmap_name)
    n_events = len(reco_data['t0'])
    colors = [cmap(0.3 + 0.6 * (i / max(n_events - 1, 1)))
              for i in range(n_events)]

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 6))
    axes = axes.flatten()

    # 🔁 Swap top-right with bottom-middle (as requested earlier)
    axes[2], axes[4] = axes[4], axes[2]

    # --- Label formatting
    def format_label(name):
        if name == 'theta': return '$\\theta$ (rad)'
        if name == 'phi': return '$\\phi$ (rad)'
        if name == 't0': return '$t_0$ (ns)'
        if name == 'E': return 'E (MeV)'
        if name in ['X', 'Y', 'Z']: return f'{name} (m)'
        return name

    # --- Param mapping
    param_map = {
        'X': ('position', 0), 'Y': ('position', 1), 'Z': ('position', 2),
        't0': ('t0', None), 'theta': ('theta', None),
        'phi': ('phi', None), 'E': ('energy', None)
    }

    def get_param_values(name, event_id):
        field, idx = param_map[name]
        arr = reco_data[field][event_id]
        return arr if idx is None else arr[:, idx]

    img_handles = []

    for i, (scan_data, ax) in enumerate(zip(scan_results_2d, axes)):

        param1_name = scan_data['param1_name']
        param2_name = scan_data['param2_name']
        param1_values = np.array(scan_data['param1_values'])
        param2_values = np.array(scan_data['param2_values'])
        losses = np.array(scan_data['losses'])
        gradients = np.array(scan_data['gradients'])
        true_value1 = scan_data['true_value1']
        true_value2 = scan_data['true_value2']

        # --- Convert to ΔLoss and ensure positive for LogNorm
        losses -= np.min(losses)
        losses += 1e-6  # Prevent log(0)

        # --- Gradient scaling
        if param1_name in ['X', 'Y', 'Z']:
            gradients[:, :, 0] *= 0.05
        if param2_name in ['X', 'Y', 'Z']:
            gradients[:, :, 1] *= 0.05
        if param1_name in ['theta', 'phi']:
            gradients[:, :, 0] *= 0.005
        if param2_name in ['theta', 'phi']:
            gradients[:, :, 1] *= 0.005
        if param1_name == 't0':
            gradients[:, :, 0] *= 0.1
        if param2_name == 'E':
            gradients[:, :, 1] *= 1000

        # --- Grid
        p1_min, p1_max = float(param1_values[0]), float(param1_values[-1])
        p2_min, p2_max = float(param2_values[0]), float(param2_values[-1])
        n_p1, n_p2 = len(param1_values), len(param2_values)
        p1_lin = np.linspace(p1_min, p1_max, n_p1)
        p2_lin = np.linspace(p2_min, p2_max, n_p2)

        # --- Normalize gradients
        U = -gradients[:, :, 0].T
        V = -gradients[:, :, 1].T
        norm = np.sqrt(U**2 + V**2)
        norm = np.where(norm == 0, 1, norm)
        U /= norm
        V /= norm

        # --- LOG SCALE LOSS MAP
        im = ax.imshow(
            losses.T,
            extent=[p1_min, p1_max, p2_min, p2_max],
            origin='lower',
            aspect='auto',
            cmap='viridis',
            norm=LogNorm(vmin=vmin, vmax=vmax)
        )
        img_handles.append(im)

        # --- Streamlines
        ax.streamplot(p1_lin, p2_lin, U, V,
                      color='white', density=1.0,
                      linewidth=0.7, arrowsize=0.5)

        # --- Optimization trajectories
        for event_id, color in enumerate(colors):
            p1_path = get_param_values(param1_name, event_id)
            p2_path = get_param_values(param2_name, event_id)

            ax.plot(p1_path, p2_path, color=color, linewidth=2)
            ax.scatter(p1_path, p2_path, color=color, s=10)
            ax.plot(p1_path[-1], p2_path[-1],
                    color='cyan', marker='x',
                    markersize=14, zorder=100+event_id)

        # --- True value
        ax.plot(true_value1, true_value2,
                color='deeppink', marker='*',
                markersize=20, zorder=99)

        ax.set_xlabel(format_label(param1_name))
        ax.set_ylabel(format_label(param2_name))
        ax.set_xlim(p1_min, p1_max)
        ax.set_ylim(p2_min, p2_max)

    # --- Shared colorbar (log scale automatically handled)
    cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
    cbar = fig.colorbar(img_handles[0], cax=cbar_ax)
    cbar.set_label('$\\Delta$ Loss (log scale)', fontsize=12)

    # --- Shared legend
    legend_elements = [
        plt.Line2D([], [], color=colors[-1], lw=2,
                   label='Optimization paths'),
        plt.Line2D([], [], color='cyan', marker='x',
                   lw=0, markersize=10,
                   label='Reconstructed'),
        plt.Line2D([], [], color='deeppink',
                   marker='*', lw=0,
                   markersize=12,
                   label='True')
    ]

    fig.legend(handles=legend_elements,
               loc='upper center',
               ncol=3, frameon=False,
               fontsize=13)

    plt.subplots_adjust(
        left=0.05,
        right=0.90,
        top=0.92,
        bottom=0.08,
        wspace=0.34,
        hspace=0.32
    )

    if save_fig:
        figures_dir = Path('figures')
        figures_dir.mkdir(parents=True, exist_ok=True)
        figname = figures_dir / '2D_Grad_All_Pairs_Grid_LOG.pdf'
        plt.savefig(figname, dpi=150, bbox_inches='tight')
        print(f"Grid figure saved as: {figname}")

    plt.show()


In [ ]:
vmin = 1e-1
# Collect ALL losses from ALL scans
all_losses = np.concatenate([
    scan_data['losses'].flatten()
    for scan_data in scan_results_2d
])

# Convert to ΔLoss like in your plotting function
all_losses = all_losses - np.min(all_losses)

# Prevent log(0)
all_losses = all_losses + 1e-6

# Choose percentile (e.g. 99th)
vmax = float(np.percentile(all_losses, 95))

plot_all_gradient_scans_grid(scan_results_2d, reco_data, save_fig=True, cmap_name='Oranges', vmin=vmin, vmax=vmax)